# Soup: проверка кандидата на Kaggle T4

Кандидат ещё не прошёл GPU-проверку. Notebook останавливается при ошибке.
Репозиторий и версия кода уже указаны. GPU T4 x2, Internet on.
Используется одна GPU. Для приватного GitHub добавьте read-only GITHUB_TOKEN
в Kaggle Secrets. HF_TOKEN необязателен. Никогда не вставляйте ключи в ячейки.

Итог: Save Version → Save & Run All, дождитесь завершения, включите public.
Ссылку с scriptVersionId добавьте в README и SUBMISSION.
Два прогона каждой стороны могут занять 20–30 минут плюс установка и загрузка.


In [ ]:
REPO_URL = "https://github.com/nazek52/soup-kernel-lab.git"  # публичный репозиторий проекта
COMMIT = "8d63b2445d68c909758679262d8bb10c8ae7d8ed"  # пусто = актуальный main; для сохранённой финальной версии укажите SHA

import re
assert re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", REPO_URL), "Заполните GitHub URL"
assert not COMMIT or re.fullmatch(r"[0-9a-f]{40}", COMMIT), "Заполните полный SHA коммита"


In [ ]:
import os
import subprocess
import sys

gpus = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True, check=True
).stdout
print(gpus)
assert all("T4" in name for name in gpus.splitlines() if name.strip()), "Выберите GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "soup-cli[train]==0.77.0", "transformers==5.16.1", "peft==0.20.0",
    "trl==0.29.1", "bitsandbytes==0.50.2", "accelerate==1.14.0", "pytest"
], check=True)


In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)  # always a fresh copy of your code
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
from pathlib import Path
WORK = Path("/kaggle/working/repo")
def run(*args):
    subprocess.run([sys.executable, *args], cwd=WORK, check=True)
run("task/prepare_data.py")
run("-m", "pytest", "solution", "-q")


In [ ]:
run("harness/run_task.py", "--repeats", "2")


In [ ]:
import json
try:
    run("harness/check_rules.py")
finally:
    # Print actual files even if a rule failed. Never manufacture PASS outputs.
    for name in ("result.json", "checks.json"):
        path = WORK / "runs" / name
        if path.exists():
            print(name)
            print(json.dumps(json.loads(path.read_text()), ensure_ascii=False, indent=2))
run("harness/score.py", "runs/result.json")


## Проверка итогов

Нужны 10/10 PASS, достаточная доля собственных ядер и сохранённые выводы.
Если есть FAIL, исправьте solution, обновите COMMIT и выполните полный Run All.
Частичный запуск не заменяет итоговую проверку.
